In [ ]:
# Import Libraries
import pandas as pd

# Load the dataset (full path to your large CSV file)
file_path = "/Users/evgeniakouklaki/Library/CloudStorage/OneDrive-Personal/Erasmus | Data Science and Marketing Analytics/Thesis/Revelio Employer Branding/urajgcxbttacmbwv.csv"

#Read file
df = pd.read_csv(file_path)

In [ ]:
#Random Sampling
df_sample = df.sample(n=100000, random_state=42)

# Print the final size of your sampled dataset
print(df_sample.shape)

# Show first 5 rows
df_sample.head()

In [ ]:
# Check the columns
df_sample.columns

In [ ]:
# Filter only english reviews
df_sample = df_sample[df_sample['review_language_id'] == 'eng']
print(df_sample.shape)

In [ ]:
# Detect language
!pip install langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42

def detect_lang(text):
    try:
        return detect(text)
    except:
        return "unknown"

df_sample["detected_lang"] = df_sample["review_summary"].apply(detect_lang)

df_sample["detected_lang"].value_counts()

In [ ]:
#Keep only rows that detect language is english
df_sample = df_sample[df_sample["detected_lang"] == "en"]

In [ ]:
# Apply detect language in all the fields
df_sample["lang_summary"] = df_sample["review_summary"].apply(detect_lang)
df_sample["lang_pros"] = df_sample["review_pros"].apply(detect_lang)
df_sample["lang_cons"] = df_sample["review_cons"].apply(detect_lang)

df_sample = df_sample[
    (df_sample["lang_summary"] == "en") &
    (df_sample["lang_pros"] == "en") &
    (df_sample["lang_cons"] == "en")
]
print(len(df_sample))

In [ ]:
# Check missining values in the text columns
text_cols = ['review_summary', 'review_pros', 'review_cons', 'review_advice']

df_sample[text_cols].isna().sum()

In [ ]:
# Drop rows with missing values in review_summary
df_sample = df_sample.dropna(subset=['review_summary'])
print(df_sample.shape)

In [ ]:
# Basic Text Cleaning
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'\n', ' ', text)  # remove new lines
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

# Apply to each text column
df_sample['summary_clean'] = df_sample['review_summary'].apply(clean_text)
df_sample['pros_clean'] = df_sample['review_pros'].apply(clean_text)
df_sample['cons_clean'] = df_sample['review_cons'].apply(clean_text)

# Check 
df_sample[['summary_clean']].head()

In [ ]:
# Exploratory Data Analysis:

# Check distribution of ratings
df_sample['rating_overall'].value_counts().sort_index()

# Visual of distribution of ratings 
import matplotlib.pyplot as plt

df_sample['rating_overall'].value_counts().sort_index().plot(kind='bar')
plt.title("Distribution of Overall Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

# Rating data frame
# Counts
rating_counts = df_sample['rating_overall'].value_counts().sort_index()

# Percentages
rating_percent = df_sample['rating_overall'].value_counts(normalize=True).sort_index() * 100

# Combine
rating_df = pd.DataFrame({
    'Count': rating_counts,
    'Percentage (%)': rating_percent.round(2)
})

print(rating_df)

In [ ]:
# Analyze review length
# Create review length columns
df_sample['summary_len'] = df_sample['review_summary'].str.split().str.len()
df_sample['pros_len'] = df_sample['review_pros'].str.split().str.len()
df_sample['cons_len'] = df_sample['review_cons'].str.split().str.len()

# Check descriptive statistics
df_sample[['summary_len', 'pros_len', 'cons_len']].describe()

In [ ]:
# Check duplicates in real data
print("Duplicate summaries in real data:", df_sample.duplicated(subset=['summary_clean']).sum())
print("Duplicate pros in real data:", df_sample.duplicated(subset=['pros_clean']).sum())
print("Duplicate cons in real data:", df_sample.duplicated(subset=['cons_clean']).sum())
print("Total real reviews:", len(df_sample))

In [ ]:
# Identify the most frequent words in each textual field-real
from collections import Counter
# --- SUMMARY ---
print("TOP WORDS - SUMMARY")
all_words_summary = " ".join(df_sample["summary_clean"].dropna()).split()
print(Counter(all_words_summary).most_common(50))


# --- PROS ---
print("\nTOP WORDS - PROS")
all_words_pros = " ".join(df_sample["pros_clean"].dropna()).split()
print(Counter(all_words_pros).most_common(50))


# --- CONS ---
print("\nTOP WORDS - CONS")
all_words_cons = " ".join(df_sample["cons_clean"].dropna()).split()
print(Counter(all_words_cons).most_common(50))

In [ ]:
#BERTopic libraries and model installation
# Install if needed
# !pip install bertopic sentence-transformers umap-learn hdbscan

# Import libraries
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import MaximalMarginalRelevance

In [ ]:
# Global Setup BERTopic-
# Sentence embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# UMAP: reduces embedding dimensions
umap_model = UMAP(
    n_neighbors=10,
    n_components=10,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN: clusters similar reviews into topics
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer: extracts words for topic representation
vectorizer_model = CountVectorizer(
    stop_words='english',
    min_df=1,
    max_df=0.85
)

# MMR: improves topic keywords by reducing redundancy
representation_model = MaximalMarginalRelevance(diversity=0.5)


In [ ]:
# Cluster Company Archetypes-real
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt


# Select rating variables

rating_vars = [
    "rating_work_life_balance",
    "rating_career_opportunities",
    "rating_compensation_and_benefits",
    "rating_senior_leadership",
    "rating_culture_and_values"
]

# Drop missing values
df_cluster = df_sample.dropna(subset=rating_vars).copy()


# Scale data

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[rating_vars])

# Find optimal number of clusters

silhouette_scores = {}

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores[k] = score

print("Silhouette scores:", silhouette_scores)

plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Choosing number of company archetypes")
plt.show()


# Run final clustering (set k based on the optimal number of clusters, 3)

k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

df_cluster["company_cluster"] = kmeans.fit_predict(X_scaled)


# Analyze clusters

cluster_profiles = df_cluster.groupby("company_cluster")[rating_vars].mean().round(2)
print("\nCluster Profiles:\n", cluster_profiles)

cluster_sizes = df_cluster["company_cluster"].value_counts(normalize=True).round(3)
print("\nCluster Sizes:\n", cluster_sizes)


# Merge back to main dataset

df_sample.loc[df_cluster.index, "company_cluster"] = df_cluster["company_cluster"]


In [ ]:
# Cluster Employee Personas-real
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt


# Select variables

persona_df = df_sample[[
    'seniority',
    'reviewer_length_of_employment',
    'reviewer_current_job'
]].copy()


# Preprocessing


# Convert boolean to int
persona_df['reviewer_current_job'] = persona_df['reviewer_current_job'].astype(int)

# Handle missing values
persona_df = persona_df.dropna()


# Scale features

scaler = StandardScaler()
X = scaler.fit_transform(persona_df)


# Find optimal clusters

sil_scores = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

# Plot
plt.plot(K_range, sil_scores, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette score')
plt.title('Choosing number of employee personas')
plt.show()

# Final clustering
k_optimal = 3  

kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
persona_df['persona_cluster'] = kmeans.fit_predict(X)

# Profile personas

persona_profiles = persona_df.groupby('persona_cluster').mean()
print("\nPersona Profiles:\n")
print(persona_profiles)

# Cluster sizes
cluster_sizes = persona_df['persona_cluster'].value_counts(normalize=True)
print("\nCluster Sizes:\n")
print(cluster_sizes)

In [ ]:
# Assign meaningfull labels to the company types clusters-real
cluster_name_map = {
    0: "Top Employer Company",
    1: "Challenging Workplace Company",
    2: "Average Workplace Company"
}

df_sample["company_archetype"] = df_sample["company_cluster"].map(cluster_name_map)

In [ ]:
 # Check the observations belonging to each company cluster
df_sample["company_archetype"].value_counts()

In [ ]:
# Assign meaningfull labels to the employee types clusters-real
df_sample.loc[persona_df.index, "persona_cluster"] = persona_df["persona_cluster"]
persona_name_map = {
    0: "Active Mid-Level Employee",
    1: "Long-Tenure Veteran",
    2: "Former Employee"
}
df_sample["employee_persona"] = df_sample["persona_cluster"].map(persona_name_map)

In [ ]:
# Check the observations belonging to each employee cluster-real
df_sample["employee_persona"].value_counts()

In [ ]:
# Check how many observations belong to each combination of company type and employee persona-real
pd.crosstab(
    df_sample["employee_persona"],
    df_sample["company_archetype"])

In [ ]:
# Build matched real sample: 500 (or max available) per persona-archetype combination
def sample_matched(df, group_cols, n=500, random_state=42):
    return (
        df.groupby(group_cols, group_keys=False)
          .apply(lambda g: g.sample(n=min(len(g), n), random_state=random_state))
    )

df_sample_matched = sample_matched(df_sample, ["employee_persona", "company_archetype"], n=500)

# Sanity check: confirm matched sample sizes
pd.crosstab(df_sample_matched["employee_persona"], df_sample_matched["company_archetype"])

In [ ]:
# Descriptive statistics of the equalised real sample (word count)
import pandas as pd

def compute_word_count_stats(df, fields):
    stats = {}
    for field in fields:
        lengths = df[field].dropna().apply(lambda x: len(str(x).split()))
        stats[field] = {
            "mean": round(lengths.mean(), 2),
            "median": round(lengths.median(), 2),
            "std": round(lengths.std(), 2),
            "min": lengths.min(),
            "max": lengths.max(),
        }
    return stats

real_fields = ["summary_clean", "pros_clean", "cons_clean"]
equalised_wc = compute_word_count_stats(df_sample_matched, real_fields)

print("=== DESCRIPTIVE STATISTICS: EQUALISED REAL SAMPLE ===\n")
for field in real_fields:
    field_name = field.replace("_clean", "")
    print(f"{field_name.upper()}")
    print(f"  Mean: {equalised_wc[field]['mean']} | Median: {equalised_wc[field]['median']} | Std: {equalised_wc[field]['std']} | Min: {equalised_wc[field]['min']} | Max: {equalised_wc[field]['max']}")
    print()

In [ ]:
len(df_sample_matched)

In [ ]:
# BERTopic summary reviews-real
# Prepare summary documents
docs_summary = df_sample_matched['summary_clean'].dropna().tolist()

# Create BERTopic model
topic_model_summary = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    calculate_probabilities=False,
    verbose=True
)
# Fit model
topics_summary, probs_summary = topic_model_summary.fit_transform(docs_summary)
#Reduce number of topics 
topic_model_summary = topic_model_summary.reduce_topics(docs_summary, nr_topics=20)
# Show topics
topic_model_summary.get_topic_info()

In [ ]:
# BERTopic pros reviwes-real
# Prepare pros documents
docs_pros = df_sample_matched['pros_clean'].dropna().tolist()
# Create BERTopic model
topic_model_pros = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    calculate_probabilities=False,
    verbose=True
)
# Fit model
topics_pros, probs_pros = topic_model_pros.fit_transform(docs_pros)
#Reduce number of topics 
topic_model_pros.reduce_topics(docs_pros, nr_topics=20)
# Show topics
topic_model_pros.get_topic_info()

In [ ]:
# BERTopic on CONS reviews-real
# Prepare cons documents
docs_cons = df_sample_matched['cons_clean'].dropna().tolist()
# Create BERTopic model
topic_model_cons = BERTopic(
    embedding_model=embedding_model,       
    umap_model=umap_model,                  
    hdbscan_model=hdbscan_model,           
    vectorizer_model=vectorizer_model,      
    representation_model=representation_model,  
    calculate_probabilities=False,
    verbose=True
)

# Fit model
topics_cons, probs_cons = topic_model_cons.fit_transform(docs_cons)
# Reduce number of topics 
topic_model_cons.reduce_topics(docs_cons, nr_topics=20)

# Show topics
topic_model_cons.get_topic_info()


In [ ]:
# Sentiment Analysis with ROBERTa-import the essential libraries-real
!pip install transformers torch -q

import pandas as pd
from transformers import pipeline
from tqdm import tqdm

In [ ]:
#Load the ROBERTa model-real
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_model = pipeline(
    "sentiment-analysis",
    model=model_path,
    tokenizer=model_path,
    truncation=True
)

In [ ]:
# Apply RoBERTa model on the matched real sample (summary field)
from tqdm import tqdm

texts_matched = df_sample_matched["summary_clean"].astype(str).tolist()

all_results_matched = []

for i in tqdm(range(0, len(texts_matched), 64)):
    batch = texts_matched[i:i+64]
    results = sentiment_model(batch, truncation=True)
    all_results_matched.extend(results)

df_sample_matched["summary_sentiment"] = [
    r["label"].lower() for r in all_results_matched
]

df_sample_matched["summary_confidence"] = [
    r["score"] for r in all_results_matched
]

In [ ]:
# Results of sentiment with RoBERTa in summary reviews-real
# Counts
sent_counts = df_sample_matched["summary_sentiment"].value_counts()

# Percentages
sent_percent = df_sample_matched["summary_sentiment"].value_counts(normalize=True) * 100

# Combine
sent_df = pd.DataFrame({
    "Count": sent_counts,
    "Percentage (%)": sent_percent.round(2)
})

print(sent_df)

In [ ]:
# Cluster Company Archetypes-real
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler


In [ ]:
# Sentiment distribution-real
summary_sentiment_dist = (
    df_sample_matched["summary_sentiment"]
    .value_counts(normalize=True)
    * 100
)

print(summary_sentiment_dist)


In [ ]:
# Visualization of sentiment distribution-real
import matplotlib.pyplot as plt

df_sample_matched["summary_sentiment"].value_counts().plot(
    kind="bar"
)

plt.title("Sentiment Distribution - Summary Reviews")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
# Average Rating per Sentiment Category to verify the sentiment-real
df_sample_matched.groupby("summary_sentiment")["rating_overall"].mean()

In [ ]:
# Use the sentiment analysis from before to identify the sentiment distribution by employee persona and company archetype-real (matched sample)
sentiment_combo = (
    df_sample_matched
    .groupby(["employee_persona", "company_archetype"])["summary_sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index(name="Percentage")
)
sentiment_combo

In [ ]:
# Easier to read sentiment per combination of employee persona and company type-real
sentiment_combo_table = sentiment_combo.pivot_table(
    index=["employee_persona", "company_archetype"],
    columns="summary_sentiment",
    values="Percentage",
    fill_value=0
)
sentiment_combo_table

In [ ]:
# Sentiments per combination and label-real
sentiment_combo_table["overall_sentiment"] = (
    sentiment_combo_table[
        ["negative", "neutral", "positive"]
    ].idxmax(axis=1)
)

sentiment_combo_table

In [ ]:
# Topic modeling per employee persona × company archetype combination-real
def run_bertopic_by_group(df, text_col, group_cols):
    results = {}
    for group_values, group_df in df.groupby(group_cols):
        docs = (
            group_df[text_col]
            .dropna()
            .astype(str)
            .tolist()
        )
        print(f"Running BERTopic for {group_values} | documents: {len(docs)}")
        try:
            topic_model = BERTopic(
                embedding_model=embedding_model,
                umap_model=umap_model,
                hdbscan_model=hdbscan_model,
                vectorizer_model=vectorizer_model,
                representation_model=representation_model,
                calculate_probabilities=False,
                verbose=False
            )
            topics, probs = topic_model.fit_transform(docs)
            topic_info = topic_model.get_topic_info()
            results[group_values] = topic_info
        except Exception as e:
            print(f"Error for {group_values}: {e}")
            continue
    return results

In [ ]:
# Show topics discussed the most in Summary review per combination-real (matched sample)
summary_topics_by_combo = run_bertopic_by_group(
    df=df_sample_matched,
    text_col="summary_clean",
    group_cols=["employee_persona", "company_archetype"]
)
# Show topics discussed the most in pros per combination-real (matched sample)
pros_topics_by_combo = run_bertopic_by_group(
    df=df_sample_matched,
    text_col="pros_clean",
    group_cols=["employee_persona", "company_archetype"]
)
# Show topics discussed the most in cons per combination-real (matched sample)
cons_topics_by_combo = run_bertopic_by_group(
    df=df_sample_matched,
    text_col="cons_clean",
    group_cols=["employee_persona", "company_archetype"]
)

In [ ]:
# Extract all topics for all combinations into one table-real for review summary-real
for combo, topic_info in summary_topics_by_combo.items():

    print("\n" + "="*100)
    print(f"Employee Persona: {combo[0]}")
    print(f"Company Archetype: {combo[1]}")
    print("="*100)

    display(
        topic_info[
            topic_info["Topic"] != -1
        ][["Topic", "Count", "Name", "Representation"]]
    )
    

In [ ]:
import pickle
with open("summary_topics_by_combo.pkl", "wb") as f:
    pickle.dump(summary_topics_by_combo, f)

In [ ]:
# Extract all topics for all combinations into one table-real for pros-real
print("\n" + "#"*100)
print("PROS TOPICS")
print("#"*100)
for combo, topic_info in pros_topics_by_combo.items():
    print("\n" + "="*100)
    print(f"Employee Persona: {combo[0]}")
    print(f"Company Archetype: {combo[1]}")
    print("="*100)
    display(
        topic_info[topic_info["Topic"] != -1]
        .sort_values("Count", ascending=False)
        .head(10)
    )

# Extract all topics for all combinations into one table-real for cons-real
print("\n" + "#"*100)
print("CONS TOPICS")
print("#"*100)
for combo, topic_info in cons_topics_by_combo.items():
    print("\n" + "="*100)
    print(f"Employee Persona: {combo[0]}")
    print(f"Company Archetype: {combo[1]}")
    print("="*100)
    display(
        topic_info[topic_info["Topic"] != -1]
        .sort_values("Count", ascending=False)
        .head(10)
    )

In [ ]:
# Define employee personas and company types-real
employee_personas = [
    "Active Mid-Level Employee",
    "Former Employee",
    "Long-Tenure Veteran"
]

company_archetypes = [
    "Top Employer Company",
    "Average Workplace Company",
    "Challenging Workplace Company"
]

In [ ]:
#Install open ai
!pip install openai

In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time
client = OpenAI(api_key="YOUR_API_KEY_HERE")

In [ ]:
# Define the generation reviews process-synthetic
def generate_reviews_batch(persona, archetype, n_reviews=10):

    prompt = f"""
You are a {persona} working for a {archetype}.

Generate {n_reviews} natural and realistic employee reviews similar to those found on Glassdoor.

Requirements:
- Use a natural and authentic tone.
- Use a writing style commonly found on employee review platforms.
- Reflect this employee persona and company archetype.
- Do not mention specific company names, industries, or personal names.
- Do not use bullet points within the reviews.
- Avoid overly formal, exaggerated, or repetitive language.
- Keep each review concise and realistic.
- Ensure that each review is unique.

Return ONLY a valid JSON object in this format:

{{
  "reviews": [
    {{
      "summary": "",
      "pros": "",
      "cons": ""
    }}
  ]
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        response_format={"type": "json_object"}
    )

    return response.choices[0].message.content

In [ ]:
# Review generation-synthetic-run only once
synthetic_dfs = {}
 
n_per_combination = 500
batch_size = 10

for persona in employee_personas:
    for archetype in company_archetypes:

        combo_name = f"{persona}__{archetype}"
        combo_reviews = []

        print(f"Generating: {combo_name}")

        for i in range(0, n_per_combination, batch_size):

            try:
                review_text = generate_reviews_batch(
                    persona,
                    archetype,
                    n_reviews=batch_size
                )

                review_json = json.loads(review_text)

                for r in review_json["reviews"]:

                    combo_reviews.append({
                        "employee_persona": persona,
                        "company_archetype": archetype,
                        "review_summary_syn": r.get("summary", ""),
                        "review_pros_syn": r.get("pros", ""),
                        "review_cons_syn": r.get("cons", "")
                    })

            except Exception as e:
                print(f"Error at {combo_name}, batch {i}: {e}")

            time.sleep(0.1)

        synthetic_dfs[combo_name] = pd.DataFrame(combo_reviews)

In [ ]:
# Merge the reviews in one df-run only once
synthetic_df = pd.concat(
    synthetic_dfs.values(),
    ignore_index=True
)

In [ ]:
# Save the df-run only once
synthetic_df.to_csv("synthetic_reviews.csv", index=False)

In [ ]:
#Load the df
synthetic_df = pd.read_csv("synthetic_reviews.csv")

In [ ]:
# Print the number of reviews generated per combination
print(synthetic_df.shape)
print(synthetic_df.groupby(['employee_persona', 'company_archetype']).size())

In [ ]:
# Clean Text in syntehtic review data-syntehtic
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

synthetic_df["summary_clean_syn"] = synthetic_df["review_summary_syn"].apply(clean_text)
synthetic_df["pros_clean_syn"] = synthetic_df["review_pros_syn"].apply(clean_text)
synthetic_df["cons_clean_syn"] = synthetic_df["review_cons_syn"].apply(clean_text)

In [ ]:
# Duplicate check on synthetic reviews-synthetic
print("=== DUPLICATE CHECK ===")
print("Duplicate summaries:", synthetic_df.duplicated(subset=['summary_clean_syn']).sum())
print("Duplicate pros:", synthetic_df.duplicated(subset=['pros_clean_syn']).sum())
print("Duplicate cons:", synthetic_df.duplicated(subset=['cons_clean_syn']).sum())
print("Total reviews:", len(synthetic_df))

In [ ]:
# Check review length of the cleaned text-synthetic
synthetic_df["summary_length"] = synthetic_df["summary_clean_syn"].apply(lambda x: len(x.split()))
synthetic_df["pros_length"] = synthetic_df["pros_clean_syn"].apply(lambda x: len(x.split()))
synthetic_df["cons_length"] = synthetic_df["cons_clean_syn"].apply(lambda x: len(x.split()))

synthetic_df[["summary_length", "pros_length", "cons_length"]].describe()

In [ ]:
# Check top words of synthetic reviews-synthetic
from collections import Counter

print("TOP WORDS - SUMMARY")
all_words_summary = " ".join(synthetic_df["summary_clean_syn"].dropna()).split()
print(Counter(all_words_summary).most_common(50))

print("\nTOP WORDS - PROS")
all_words_pros = " ".join(synthetic_df["pros_clean_syn"].dropna()).split()
print(Counter(all_words_pros).most_common(50))

print("\nTOP WORDS - CONS")
all_words_cons = " ".join(synthetic_df["cons_clean_syn"].dropna()).split()
print(Counter(all_words_cons).most_common(50))

In [ ]:
# Apply RoBERTa sentiment analysis to all synthetic review summaries-syntehtic
from tqdm import tqdm

texts_syn = synthetic_df["summary_clean_syn"].astype(str).tolist()
all_results_syn = []
for i in tqdm(range(0, len(texts_syn), 64)):
    batch = texts_syn[i:i+64]
    results = sentiment_model(batch, truncation=True)
    all_results_syn.extend(results)

synthetic_df["summary_sentiment"] = [r["label"].lower() for r in all_results_syn]
synthetic_df["summary_confidence"] = [r["score"] for r in all_results_syn]

In [ ]:
# Overall sentiment distribution of synthetic reviews in review summary-syntehtic

sentiment_counts = synthetic_df["summary_sentiment"].value_counts()
sentiment_percent = synthetic_df["summary_sentiment"].value_counts(normalize=True) * 100

sentiment_table = pd.DataFrame({
    "Count": sentiment_counts,
    "Percentage": sentiment_percent.round(2)
})

sentiment_table

In [ ]:
# Visualize overall sentiment distribution-syntehtic
import matplotlib.pyplot as plt
sentiment_counts.plot(kind="bar")

plt.title("Synthetic Review Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

plt.show()

In [ ]:
# Sentiment distribution by employee persona and company archetype-synthetic
sentiment_combo_table_syn = pd.crosstab(
    [synthetic_df["employee_persona"],
     synthetic_df["company_archetype"]],
    synthetic_df["summary_sentiment"],
    normalize="index"
) * 100

sentiment_combo_table_syn.round(2)

In [ ]:
# Assign overall sentiment label

sentiment_combo_table_syn["overall_sentiment"] = (
    sentiment_combo_table_syn[
        ["negative", "neutral", "positive"]
    ]
    .idxmax(axis=1)
)

sentiment_combo_table_syn

In [ ]:
# BERTopic to identify topics discussed in synthetic reviews
def run_bertopic(docs, name):

    # BERTopic model
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        verbose=True
    )

    # Fit model
    topics, probs = topic_model.fit_transform(docs)

    # Reduce number of topics
    topic_model = topic_model.reduce_topics(
        docs,
        nr_topics=20
    )

    # Show topics
    print(f"\n{name} TOPICS")
    display(topic_model.get_topic_info())
    
    return topic_model, topics

In [ ]:
# BERTopic on synthetic review summaries-synthetic

summary_docs_syn = synthetic_df["summary_clean_syn"].dropna().tolist()

summary_topic_model_syn, summary_topics_syn = run_bertopic(
    summary_docs_syn,
    "SYNTHETIC SUMMARY"
)

In [ ]:
#BERTopic on synthetic review pros-synthetic

pros_docs_syn = synthetic_df["pros_clean_syn"].dropna().tolist()

pros_topic_model_syn, pros_topics_syn = run_bertopic(
    pros_docs_syn,
    "SYNTHETIC PROS"
)

In [ ]:
# BERTopic on synthetic review cons-synthetic

cons_docs_syn = synthetic_df["cons_clean_syn"].dropna().tolist()

cons_topic_model_syn, cons_topics_syn = run_bertopic(
    cons_docs_syn,
    "SYNTHETIC CONS"
)

In [ ]:
#Run BERTopic on Synthetic Summaries by Combination-synthetic
summary_topics_by_combo_syn = run_bertopic_by_group(
    df=synthetic_df,
    text_col="summary_clean_syn",
    group_cols=["employee_persona", "company_archetype"],
)

In [ ]:
#Display topics for all combinations on summary-synthetic
for combo, topic_info in summary_topics_by_combo_syn.items():

    print("\n" + "="*100)
    print(f"Employee Persona: {combo[0]}")
    print(f"Company Archetype: {combo[1]}")
    print("="*100)

    display(
        topic_info[
            topic_info["Topic"] != -1
        ][["Topic", "Count", "Name", "Representation"]]
    )

In [ ]:
# Run BERTopic on Synthetic Pros by Combination-synthetic
pros_topics_by_combo_syn = run_bertopic_by_group(
    df=synthetic_df,
    text_col="pros_clean_syn",
    group_cols=["employee_persona", "company_archetype"]
)

# Run BERTopic on Synthetic Cons by Combination
cons_topics_by_combo_syn = run_bertopic_by_group(
    df=synthetic_df,
    text_col="cons_clean_syn",
    group_cols=["employee_persona", "company_archetype"]
)

In [ ]:
# Display topics for all combinations - pros - synthetic
for combo, topic_info in pros_topics_by_combo_syn.items():
    print("\n" + "="*100)
    print(f"Employee Persona: {combo[0]}")
    print(f"Company Archetype: {combo[1]}")
    print("="*100)
    display(
        topic_info[
            topic_info["Topic"] != -1
        ][["Topic", "Count", "Name", "Representation"]]
    )

In [ ]:
# Display topics for all combinations - cons - synthetic
for combo, topic_info in cons_topics_by_combo_syn.items():
    print("\n" + "="*100)
    print(f"Employee Persona: {combo[0]}")
    print(f"Company Archetype: {combo[1]}")
    print("="*100)
    display(
        topic_info[
            topic_info["Topic"] != -1
        ][["Topic", "Count", "Name", "Representation"]]
    )

In [ ]:
# LLM-based analysis function for employee reviews-LLM
import json
import pandas as pd

def llm_analyze_reviews(reviews_text, group_name):
    prompt = f"""
You are an experienced qualitative research analyst.
You are analyzing employee reviews for the group: {group_name}.
Your task is to extract employer branding insights from the reviews.
Tasks:
1. Identify the main recurring themes.
2. Summarize the main strengths mentioned by employees.
3. Summarize the main weaknesses mentioned by employees.
4. Describe the overall sentiment in a short sentence.
5. Provide a short employer branding interpretation.
6. Provide a single overall sentiment label that best summarizes point 4: choose exactly one of "positive", "neutral", or "negative".

Reviews:
{reviews_text}

Return ONLY valid JSON in this format:
{{
  "group": "{group_name}",
  "main_themes": [],
  "strengths": [],
  "weaknesses": [],
  "overall_sentiment": "",
  "overall_sentiment_label": "",
  "employer_branding_interpretation": ""
}}
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# Prepare review text for LLM analysis (150 observations per combination)-LLM

def prepare_reviews_text(df, n=150):

    sample_df = df.sample(
        n=min(n, len(df)),
        random_state=42
    )

    reviews_text = "\n\n".join(
        "Summary: " + sample_df["review_summary_syn"].astype(str) +
        "\nPros: " + sample_df["review_pros_syn"].astype(str) +
        "\nCons: " + sample_df["review_cons_syn"].astype(str)
    )

    return reviews_text

In [ ]:
# Scenario 3A: LLM-based analysis of all synthetic reviews-LLM

all_reviews_text = prepare_reviews_text(
    synthetic_df,
    n=150
)

llm_all_results = llm_analyze_reviews(
    all_reviews_text,
    group_name="All Synthetic Reviews"
)

llm_all_results

In [ ]:
# Scenario 3D: LLM-based analysis by employee persona and company archetype combination-LLM
llm_combo_results = []
for (persona, archetype), group_df in synthetic_df.groupby(
    ["employee_persona", "company_archetype"]
):
    # Prepare review text sample for this combination
    # n=150 reviews per group as specified in the methodology
    reviews_text = prepare_reviews_text(
        group_df,
        n=150
    )
    # Run LLM-based analysis for this persona-archetype combination
    result = llm_analyze_reviews(
        reviews_text,
        group_name=f"{persona} + {archetype}"
    )
    # Tag each result with its persona and archetype for traceability
    result["employee_persona"] = persona
    result["company_archetype"] = archetype
    llm_combo_results.append(result)

# Combine all results into a single dataframe
llm_combo_results_df = pd.DataFrame(llm_combo_results)

# Print full results without truncation
for idx, row in llm_combo_results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Persona: {row['employee_persona']}")
    print(f"Archetype: {row['company_archetype']}")
    print(f"Overall sentiment: {row['overall_sentiment']}")
    print(f"Overall sentiment label: {row['overall_sentiment_label']}")
    print(f"\nMain themes:")
    for theme in row['main_themes']:
        print(f"  - {theme}")
    print(f"\nStrengths:")
    for s in row['strengths']:
        print(f"  - {s}")
    print(f"\nWeaknesses:")
    for w in row['weaknesses']:
        print(f"  - {w}")
    print(f"\nEmployer branding interpretation:")
    print(f"  {row['employer_branding_interpretation']}")

In [ ]:
# Save LLM-based analysis outputs-LLM
llm_all_results_df = pd.DataFrame([llm_all_results])

llm_all_results_df.to_csv("llm_analysis_all_synthetic_reviews.csv", index=False)
llm_combo_results_df.to_csv("llm_analysis_by_combination.csv", index=False)

In [ ]:
# Print outputs-LLM
llm_all_results_df

In [ ]:
# More clear output-LLM
for col in llm_all_results_df.columns:
    print(f"\n{'='*60}")
    print(f"{col.upper()}:")
    print(llm_all_results_df[col].values[0])

In [ ]:
# Word count comparison between real and synthetic reviews-Comparison
# Assesses surface-level textual similarity 
import pandas as pd

def compute_word_count_stats(df, fields):
    stats = {}
    for field in fields:
        lengths = df[field].dropna().apply(lambda x: len(str(x).split()))
        stats[field] = {
            "mean": round(lengths.mean(), 2),
            "median": round(lengths.median(), 2),
            "std": round(lengths.std(), 2)
        }
    return stats

real_fields = ["summary_clean", "pros_clean", "cons_clean"]
syn_fields = ["summary_clean_syn", "pros_clean_syn", "cons_clean_syn"]

real_wc = compute_word_count_stats(df_sample_matched, real_fields)
syn_wc = compute_word_count_stats(synthetic_df, syn_fields)

print("=== WORD COUNT COMPARISON (C1 vs C2, matched sample) ===\n")
for real_f, syn_f in zip(real_fields, syn_fields):
    field_name = real_f.replace("_clean", "").replace("_syn", "")
    print(f"{field_name.upper()}")
    print(f"  Real      — Mean: {real_wc[real_f]['mean']} | Median: {real_wc[real_f]['median']} | Std: {real_wc[real_f]['std']}")
    print(f"  Synthetic — Mean: {syn_wc[syn_f]['mean']} | Median: {syn_wc[syn_f]['median']} | Std: {syn_wc[syn_f]['std']}")
    print()

In [ ]:
# Jaccard similarity between most frequent terms in real and synthetic reviews-Comparison
# Measures vocabulary overlap between datasets 
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def compute_jaccard(df_real, df_syn, field_real, field_syn, top_n=50):
    
    real_counter = Counter(
        w for w in " ".join(df_real[field_real].dropna()).split()
        if w.lower() not in ENGLISH_STOP_WORDS and len(w) > 2
    )
    syn_counter = Counter(
        w for w in " ".join(df_syn[field_syn].dropna()).split()
        if w.lower() not in ENGLISH_STOP_WORDS and len(w) > 2
    )
    
    real_words = real_counter.most_common(top_n)
    syn_words = syn_counter.most_common(top_n)
    
    real_set = set([w[0] for w in real_words])
    syn_set = set([w[0] for w in syn_words])
    
    intersection = len(real_set & syn_set)
    union = len(real_set | syn_set)
    jaccard = intersection / union
    
    # Sort overlapping/unique terms by their combined or individual frequency, not alphabetically
    overlapping_sorted = sorted(
        real_set & syn_set,
        key=lambda w: real_counter[w] + syn_counter[w],
        reverse=True
    )
    unique_to_real_sorted = sorted(
        real_set - syn_set,
        key=lambda w: real_counter[w],
        reverse=True
    )
    unique_to_synthetic_sorted = sorted(
        syn_set - real_set,
        key=lambda w: syn_counter[w],
        reverse=True
    )
    
    return {
        "jaccard_similarity": round(jaccard, 4),
        "overlapping_terms": overlapping_sorted,
        "unique_to_real": unique_to_real_sorted,
        "unique_to_synthetic": unique_to_synthetic_sorted
    }

print("=== JACCARD TERM OVERLAP (C1 vs C2) ===\n")
for real_f, syn_f in zip(real_fields, syn_fields):
    field_name = real_f.replace("_clean", "")
    result = compute_jaccard(df_sample_matched, synthetic_df, real_f, syn_f)
    print(f"{field_name.upper()}")
    print(f"  Jaccard Similarity: {result['jaccard_similarity']}")
    print(f"  Overlapping terms ({len(result['overlapping_terms'])}): {result['overlapping_terms'][:10]}...")
    print(f"  Unique to real ({len(result['unique_to_real'])}): {result['unique_to_real'][:5]}...")
    print(f"  Unique to synthetic ({len(result['unique_to_synthetic'])}): {result['unique_to_synthetic'][:5]}...")
    print()

In [ ]:
# Jensen-Shannon Divergence between sentiment distributions-Comparison
# Measures how similar sentiment distributions are across scenarios (

from scipy.spatial.distance import jensenshannon
import numpy as np

def compute_sentiment_jsd(real_sentiments, compare_sentiments, label):
    
    labels = ["positive", "neutral", "negative"]
    
    real_dist = np.array([(real_sentiments == l).mean() for l in labels])
    compare_dist = np.array([(compare_sentiments == l).mean() for l in labels])
    
    jsd = jensenshannon(real_dist, compare_dist)
    
    print(f"  {label}")
    print(f"  Real      — Positive: {real_dist[0]:.3f} | Neutral: {real_dist[1]:.3f} | Negative: {real_dist[2]:.3f}")
    print(f"  Compare   — Positive: {compare_dist[0]:.3f} | Neutral: {compare_dist[1]:.3f} | Negative: {compare_dist[2]:.3f}")
    print(f"  Jensen-Shannon Divergence: {round(jsd, 4)} (0=identical, 1=completely different)")
    print()
    
    return jsd

print("=== SENTIMENT DISTRIBUTION COMPARISON (C1 vs C2 vs C3) ===\n")

# C1 vs C2 — real vs synthetic (traditional analytics)
jsd_c1_c2 = compute_sentiment_jsd(
    df_sample_matched["summary_sentiment"],
    synthetic_df["summary_sentiment"],
    "C1 (Real, matched) vs C2 (Synthetic)"
)

In [ ]:
# Cosine similarity between topic keywords and proportion of theme recovery-Comparison
# Measures semantic similarity of topics between real and synthetic reviews


from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

def compute_topic_similarity(real_model, syn_model, field_name):
    
    # Get topic keywords excluding outlier topic (-1)
    real_topics = real_model.get_topic_info()
    real_topics = real_topics[real_topics["Topic"] != -1]["Representation"].tolist()
    
    syn_topics = syn_model.get_topic_info()
    syn_topics = syn_topics[syn_topics["Topic"] != -1]["Representation"].tolist()
    
    # Convert keyword lists to strings
    real_keywords = [" ".join(t) for t in real_topics]
    syn_keywords = [" ".join(t) for t in syn_topics]
    
    # Use semantic embeddings for meaningful topic comparison
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    real_vecs = embed_model.encode(real_keywords)
    syn_vecs = embed_model.encode(syn_keywords)
    
    # Compute cosine similarity matrix
    similarity_matrix = cosine_similarity(real_vecs, syn_vecs)
    
    # Average maximum cosine similarity per real topic
    avg_cosine = similarity_matrix.max(axis=1).mean()
    
    # Theme recovery: % of real topics with cosine >= 0.5
    recovered = sum(similarity_matrix.max(axis=1) >= 0.5)
    theme_recovery = recovered / len(real_keywords)
    
    # New themes in synthetic not in real
    new_themes = sum(similarity_matrix.max(axis=0) < 0.5)
    new_themes_pct = new_themes / len(syn_keywords)
    
    print(f"=== {field_name.upper()} ===")
    print(f"  Real topics: {len(real_keywords)} | Synthetic topics: {len(syn_keywords)}")
    print(f"  Average cosine similarity: {round(avg_cosine, 4)}")
    print(f"  Theme recovery: {round(theme_recovery*100, 1)}% of real topics recovered in synthetic")
    print(f"  New themes in synthetic: {round(new_themes_pct*100, 1)}%")
    print()
    
    return {
        "field": field_name,
        "avg_cosine_similarity": round(avg_cosine, 4),
        "theme_recovery": round(theme_recovery, 4),
        "new_themes_pct": round(new_themes_pct, 4)
    }

print("=== TOPIC SIMILARITY (C1 vs C2) ===\n")

results_summary = compute_topic_similarity(topic_model_summary, summary_topic_model_syn, "summary")
results_pros = compute_topic_similarity(topic_model_pros, pros_topic_model_syn, "pros")
results_cons = compute_topic_similarity(topic_model_cons, cons_topic_model_syn, "cons")

In [ ]:
# C3 vs C1 and C2: Final Comparison- Comparison
# 1. Cosine similarity + theme recovery to identify if themes are themes similar
# 2. Sentiment agreement rate (to identify if sentiment direction is correct?

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load sentence embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# =============================================================
# C3 themes, strengths and weaknesses
# Extracted by GPT-4o-mini from all synthetic reviews combined
# =============================================================

c3_all = [
    # Main themes
    "Work-life balance",
    "Team camaraderie and support",
    "Management issues",
    "Communication challenges",
    "Opportunities for growth",
    "Workload and stress levels",
    "Innovation and creativity",
    "Training and development",
    # Strengths
    "Friendly and supportive colleagues",
    "Flexible work hours",
    "Opportunities for professional development",
    "Engaging and challenging projects",
    "Strong team spirit and camaraderie",
    "Decent benefits package",
    # Weaknesses
    "Management often disconnected from employees",
    "Communication issues between departments",
    "High workload and stress levels",
    "Limited opportunities for advancement",
    "Lack of recognition for hard work",
    "Resistance to change and innovation"
]

# =============================================================
# METRIC 1: Cosine similarity + theme recovery
# =============================================================

def compare_c3_to_bertopic(c3_texts, topic_model, label):
    topic_info = topic_model.get_topic_info()
    topic_info = topic_info[topic_info["Topic"] != -1]["Representation"].tolist()
    topic_keywords = [" ".join(t) for t in topic_info]
    
    c3_vecs = embed_model.encode(c3_texts)
    topic_vecs = embed_model.encode(topic_keywords)
    
    sim_matrix = cosine_similarity(c3_vecs, topic_vecs)
    
    avg_cosine = sim_matrix.max(axis=1).mean()
    recovery = (sim_matrix.max(axis=1) >= 0.5).mean()
    not_captured = (sim_matrix.max(axis=0) < 0.5).mean()
    
    print(f"  C3 vs {label}:")
    print(f"    Average cosine similarity: {round(avg_cosine, 4)}")
    print(f"    C3 theme recovery: {round(recovery*100, 1)}%")
    print(f"    {label} topics NOT captured by C3: {round(not_captured*100, 1)}%")
    print()

print("=" * 60)
print("METRIC 1: COSINE SIMILARITY AND THEME RECOVERY")
print("=" * 60)

print("\n--- Summary Field ---")
compare_c3_to_bertopic(c3_all, topic_model_summary, "C1")
compare_c3_to_bertopic(c3_all, summary_topic_model_syn, "C2")

print("--- Pros Field ---")
compare_c3_to_bertopic(c3_all, topic_model_pros, "C1")
compare_c3_to_bertopic(c3_all, pros_topic_model_syn, "C2")

print("--- Cons Field ---")
compare_c3_to_bertopic(c3_all, topic_model_cons, "C1")
compare_c3_to_bertopic(c3_all, cons_topic_model_syn, "C2")

# =============================================================
# METRIC 2: Sentiment agreement rate
# =============================================================

# C3 — from updated LLM output (Scenario 3D, overall_sentiment_label), synthetic-based
c3_sentiment = {
    ("Active Mid-Level Employee", "Average Workplace Company"): "neutral",
    ("Active Mid-Level Employee", "Challenging Workplace Company"): "negative",
    ("Active Mid-Level Employee", "Top Employer Company"): "positive",
    ("Former Employee", "Average Workplace Company"): "neutral",
    ("Former Employee", "Challenging Workplace Company"): "negative",
    ("Former Employee", "Top Employer Company"): "positive",
    ("Long-Tenure Veteran", "Average Workplace Company"): "neutral",
    ("Long-Tenure Veteran", "Challenging Workplace Company"): "negative",
    ("Long-Tenure Veteran", "Top Employer Company"): "positive",
}

# C1 — from matched real sentiment table (df_sample_matched, sentiment_combo_table)
c1_sentiment = {
    ("Active Mid-Level Employee", "Average Workplace Company"): "positive",
    ("Active Mid-Level Employee", "Challenging Workplace Company"): "negative",
    ("Active Mid-Level Employee", "Top Employer Company"): "positive",
    ("Former Employee", "Average Workplace Company"): "positive",
    ("Former Employee", "Challenging Workplace Company"): "negative",
    ("Former Employee", "Top Employer Company"): "positive",
    ("Long-Tenure Veteran", "Average Workplace Company"): "positive",
    ("Long-Tenure Veteran", "Challenging Workplace Company"): "negative",
    ("Long-Tenure Veteran", "Top Employer Company"): "positive",
}

# C2 — synthetic sentiment, unaffected by real-data matching fix, unchanged
c2_sentiment = {
    ("Active Mid-Level Employee", "Average Workplace Company"): "positive",
    ("Active Mid-Level Employee", "Challenging Workplace Company"): "neutral",
    ("Active Mid-Level Employee", "Top Employer Company"): "positive",
    ("Former Employee", "Average Workplace Company"): "neutral",
    ("Former Employee", "Challenging Workplace Company"): "neutral",
    ("Former Employee", "Top Employer Company"): "positive",
    ("Long-Tenure Veteran", "Average Workplace Company"): "positive",
    ("Long-Tenure Veteran", "Challenging Workplace Company"): "neutral",
    ("Long-Tenure Veteran", "Top Employer Company"): "positive",
}

combos = list(c3_sentiment.keys())

c3_c1_agree = sum(c3_sentiment[k] == c1_sentiment[k] for k in combos)
c3_c2_agree = sum(c3_sentiment[k] == c2_sentiment[k] for k in combos)
c1_c2_agree = sum(c1_sentiment[k] == c2_sentiment[k] for k in combos)

print("=" * 60)
print("METRIC 2: SENTIMENT AGREEMENT RATE")
print("=" * 60)
print(f"\nC3 vs C1: {c3_c1_agree}/{len(combos)} agree ({c3_c1_agree/len(combos)*100:.1f}%)")
print(f"C3 vs C2: {c3_c2_agree}/{len(combos)} agree ({c3_c2_agree/len(combos)*100:.1f}%)")
print(f"C1 vs C2 benchmark: {c1_c2_agree}/{len(combos)} agree ({c1_c2_agree/len(combos)*100:.1f}%)")

print("\n--- Per Combination Breakdown ---\n")
print(f"{'Persona':<32} {'Archetype':<30} {'C1':<12} {'C2':<12} {'C3':<12} {'C3=C1':<8} {'C3=C2'}")
print("-" * 110)
for k in combos:
    persona, archetype = k
    c1 = c1_sentiment[k]
    c2 = c2_sentiment[k]
    c3 = c3_sentiment[k]
    print(f"{persona:<32} {archetype:<30} {c1:<12} {c2:<12} {c3:<12} {'Yes' if c3==c1 else 'No':<8} {'Yes' if c3==c2 else 'No'}")

In [ ]:
# Save the topics
import pickle

# Save real subgroup topics
with open("summary_topics_by_combo.pkl", "wb") as f:
    pickle.dump(summary_topics_by_combo, f)

with open("pros_topics_by_combo.pkl", "wb") as f:
    pickle.dump(pros_topics_by_combo, f)

with open("cons_topics_by_combo.pkl", "wb") as f:
    pickle.dump(cons_topics_by_combo, f)

# Save synthetic subgroup topics
with open("summary_topics_by_combo_syn.pkl", "wb") as f:
    pickle.dump(summary_topics_by_combo_syn, f)

with open("pros_topics_by_combo_syn.pkl", "wb") as f:
    pickle.dump(pros_topics_by_combo_syn, f)

with open("cons_topics_by_combo_syn.pkl", "wb") as f:
    pickle.dump(cons_topics_by_combo_syn, f)

# Save overall real topic models
topic_model_summary.save("topic_model_summary")
topic_model_pros.save("topic_model_pros")
topic_model_cons.save("topic_model_cons")

# Save overall synthetic topic models
summary_topic_model_syn.save("topic_model_summary_syn")
pros_topic_model_syn.save("topic_model_pros_syn")
cons_topic_model_syn.save("topic_model_cons_syn")

print("All models saved successfully.")

In [ ]:
#Restore the topics
import pickle

with open("summary_topics_by_combo.pkl", "rb") as f:
    summary_topics_by_combo = pickle.load(f)

with open("pros_topics_by_combo.pkl", "rb") as f:
    pros_topics_by_combo = pickle.load(f)

with open("cons_topics_by_combo.pkl", "rb") as f:
    cons_topics_by_combo = pickle.load(f)

with open("summary_topics_by_combo_syn.pkl", "rb") as f:
    summary_topics_by_combo_syn = pickle.load(f)

with open("pros_topics_by_combo_syn.pkl", "rb") as f:
    pros_topics_by_combo_syn = pickle.load(f)

with open("cons_topics_by_combo_syn.pkl", "rb") as f:
    cons_topics_by_combo_syn = pickle.load(f)

In [ ]:
# Robustness Check
# PART 1: SETUP
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import jensenshannon

group_cols = ["employee_persona", "company_archetype"]

SENTIMENT_SEEDS = list(range(1, 11))   # 10 seeds - cheap, sentiment only
TOPIC_AGG_SEEDS = list(range(1, 6))    # 5 seeds - aggregate topics
TOPIC_SUB_SEEDS = list(range(1, 4))    # 3 seeds - subgroup topics (expensive: 9x cost)

In [ ]:
# PART 2: SENTIMENT ROBUSTNESS - AGGREGATE (10 seeds)
# Averages the aggregate sentiment distribution and JSD vs synthetic
rc_agg_sentiment_runs = []

for seed in SENTIMENT_SEEDS:
    df_run = sample_matched(df_sample, group_cols, n=500, random_state=seed)

    texts = df_run["summary_clean"].astype(str).tolist()
    results = []
    for i in range(0, len(texts), 64):
        results.extend(sentiment_model(texts[i:i+64], truncation=True))
    df_run["summary_sentiment"] = [r["label"].lower() for r in results]

    dist = df_run["summary_sentiment"].value_counts(normalize=True) * 100
    labels = ["positive", "neutral", "negative"]
    real_dist = np.array([(df_run["summary_sentiment"] == l).mean() for l in labels])
    syn_dist = np.array([(synthetic_df["summary_sentiment"] == l).mean() for l in labels])
    jsd = jensenshannon(real_dist, syn_dist)

    rc_agg_sentiment_runs.append({
        "seed": seed,
        "positive": dist.get("positive", 0),
        "neutral": dist.get("neutral", 0),
        "negative": dist.get("negative", 0),
        "jsd": jsd,
    })
    print(f"  Seed {seed}: pos={dist.get('positive',0):.2f}% neu={dist.get('neutral',0):.2f}% neg={dist.get('negative',0):.2f}% JSD={jsd:.4f}")

rc_agg_sentiment_df = pd.DataFrame(rc_agg_sentiment_runs)

print("\n=== AGGREGATE SENTIMENT - AVERAGED ACROSS", len(SENTIMENT_SEEDS), "SEEDS ===")
print(f"Positive: {rc_agg_sentiment_df['positive'].mean():.2f}% ± {rc_agg_sentiment_df['positive'].std():.2f}")
print(f"Neutral:  {rc_agg_sentiment_df['neutral'].mean():.2f}% ± {rc_agg_sentiment_df['neutral'].std():.2f}")
print(f"Negative: {rc_agg_sentiment_df['negative'].mean():.2f}% ± {rc_agg_sentiment_df['negative'].std():.2f}")
print(f"JSD:      {rc_agg_sentiment_df['jsd'].mean():.4f} ± {rc_agg_sentiment_df['jsd'].std():.4f}")

In [ ]:
# PART 3: SENTIMENT ROBUSTNESS - SUBGROUP (same 10 seeds, per persona x archetype)
# Averages sentiment % per combination across seeds
rc_subgroup_sentiment_runs = []

for seed in SENTIMENT_SEEDS:
    df_run = sample_matched(df_sample, group_cols, n=500, random_state=seed)

    texts = df_run["summary_clean"].astype(str).tolist()
    results = []
    for i in range(0, len(texts), 64):
        results.extend(sentiment_model(texts[i:i+64], truncation=True))
    df_run["summary_sentiment"] = [r["label"].lower() for r in results]

    sub = (
        df_run.groupby(group_cols)["summary_sentiment"]
        .value_counts(normalize=True).mul(100)
        .rename("pct").reset_index()
    )
    sub["seed"] = seed
    rc_subgroup_sentiment_runs.append(sub)

rc_subgroup_sentiment_df = pd.concat(rc_subgroup_sentiment_runs, ignore_index=True)

rc_subgroup_avg = (
    rc_subgroup_sentiment_df
    .groupby(group_cols + ["summary_sentiment"])["pct"]
    .agg(["mean", "std"]).round(2).reset_index()
)

print("\n=== SUBGROUP SENTIMENT - AVERAGED ACROSS", len(SENTIMENT_SEEDS), "SEEDS ===")
print(rc_subgroup_avg.to_string(index=False))

In [ ]:
# PART 4: TOPIC ROBUSTNESS - AGGREGATE (5 seeds, summary/pros/cons)
# Refits BERTopic per seed, compares to synthetic, averages cosine, theme recovery, and new themes
rc_topic_agg_runs = {}
for field, syn_model in [
    ("summary", summary_topic_model_syn),
    ("pros", pros_topic_model_syn),
    ("cons", cons_topic_model_syn),
]:
    field_col = f"{field}_clean"
    results = []
    syn_topics = syn_model.get_topic_info()
    syn_topics = syn_topics[syn_topics["Topic"] != -1]["Representation"].tolist()
    syn_keywords = [" ".join(t) for t in syn_topics]
    syn_vecs = embed_model.encode(syn_keywords)
    for seed in TOPIC_AGG_SEEDS:
        df_run = sample_matched(df_sample, group_cols, n=500, random_state=seed)
        docs = df_run[field_col].dropna().tolist()
        m = BERTopic(
            embedding_model=embedding_model, umap_model=umap_model,
            hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
            representation_model=representation_model,
            calculate_probabilities=False, verbose=False
        )
        m.fit_transform(docs)
        m = m.reduce_topics(docs, nr_topics=20)
        topics = m.get_topic_info()
        topics = topics[topics["Topic"] != -1]["Representation"].tolist()
        keywords = [" ".join(t) for t in topics]
        vecs = embed_model.encode(keywords)
        sim = cosine_similarity(vecs, syn_vecs)
        avg_cosine = sim.max(axis=1).mean()
        recovery = (sim.max(axis=1) >= 0.5).mean() * 100
        new_themes = (sim.max(axis=0) < 0.5).mean() * 100
        results.append({
            "seed": seed,
            "avg_cosine_vs_synthetic": avg_cosine,
            "theme_recovery_pct": recovery,
            "new_themes_pct": new_themes,
        })
        print(f"  {field.upper()} seed {seed}: cosine={avg_cosine:.4f}, recovery={recovery:.1f}%, new_themes={new_themes:.1f}%")
    rc_topic_agg_runs[field] = pd.DataFrame(results)
print("\n=== AGGREGATE TOPICS - AVERAGED ACROSS", len(TOPIC_AGG_SEEDS), "SEEDS ===")
for field, df_r in rc_topic_agg_runs.items():
    print(f"{field.upper()}: cosine={df_r['avg_cosine_vs_synthetic'].mean():.4f} ± {df_r['avg_cosine_vs_synthetic'].std():.4f} | "
          f"recovery={df_r['theme_recovery_pct'].mean():.1f}% ± {df_r['theme_recovery_pct'].std():.1f}% | "
          f"new_themes={df_r['new_themes_pct'].mean():.1f}% ± {df_r['new_themes_pct'].std():.1f}%")

In [ ]:
# PART 5: TOPIC ROBUSTNESS - SUBGROUP (3 seeds, 9 combinations x 3 fields)
# EXPENSIVE: 81 BERTopic fits total. Run separately, expect long runtime.
rc_topic_sub_runs = []

for field, syn_combo_dict in [
    ("summary", summary_topics_by_combo_syn),
    ("pros", pros_topics_by_combo_syn),
    ("cons", cons_topics_by_combo_syn),
]:
    field_col = f"{field}_clean"

    for seed in TOPIC_SUB_SEEDS:
        df_run = sample_matched(df_sample, group_cols, n=500, random_state=seed)

        for combo_key, syn_topic_info in syn_combo_dict.items():
            persona, archetype = combo_key
            group_df = df_run[
                (df_run["employee_persona"] == persona) &
                (df_run["company_archetype"] == archetype)
            ]
            docs = group_df[field_col].dropna().tolist()
            if len(docs) < 10:
                continue

            m = BERTopic(
                embedding_model=embedding_model, umap_model=umap_model,
                hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                representation_model=representation_model,
                calculate_probabilities=False, verbose=False
            )
            try:
                m.fit_transform(docs)
            except Exception as e:
                print(f"  Skipped {combo_key} seed {seed}: {e}")
                continue

            topics = m.get_topic_info()
            topics = topics[topics["Topic"] != -1]["Representation"].tolist()
            if not topics:
                continue
            keywords = [" ".join(t) for t in topics]
            vecs = embed_model.encode(keywords)

            syn_topics = syn_topic_info[syn_topic_info["Topic"] != -1]["Representation"].tolist()
            syn_keywords = [" ".join(t) for t in syn_topics]
            syn_vecs = embed_model.encode(syn_keywords)

            sim = cosine_similarity(vecs, syn_vecs)
            avg_cosine = sim.max(axis=1).mean()

            rc_topic_sub_runs.append({
                "field": field, "persona": persona, "archetype": archetype,
                "seed": seed, "avg_cosine_vs_synthetic": avg_cosine
            })
            print(f"  {field.upper()} | {persona} + {archetype} | seed {seed}: cosine={avg_cosine:.4f}")

rc_topic_sub_df = pd.DataFrame(rc_topic_sub_runs)

print("\n=== SUBGROUP TOPICS - AVERAGED ACROSS", len(TOPIC_SUB_SEEDS), "SEEDS ===")
rc_topic_sub_avg = (
    rc_topic_sub_df
    .groupby(["field", "persona", "archetype"])["avg_cosine_vs_synthetic"]
    .agg(["mean", "std"]).round(4).reset_index()
)
print(rc_topic_sub_avg.to_string(index=False))

In [ ]:
# PART 6: WORD COUNT ROBUSTNESS (10 seeds)
# Averages mean/median/std of review length across repeated matched samples

WORDCOUNT_SEEDS = list(range(1, 11))
real_fields = ["summary_clean", "pros_clean", "cons_clean"]

rc_wordcount_runs = []

for seed in WORDCOUNT_SEEDS:
    df_run = sample_matched(df_sample, group_cols, n=500, random_state=seed)
    stats = compute_word_count_stats(df_run, real_fields)

    row = {"seed": seed}
    for field in real_fields:
        row[f"{field}_mean"] = stats[field]["mean"]
        row[f"{field}_median"] = stats[field]["median"]
        row[f"{field}_std"] = stats[field]["std"]
    rc_wordcount_runs.append(row)

rc_wordcount_df = pd.DataFrame(rc_wordcount_runs)

print("=== WORD COUNT - AVERAGED ACROSS", len(WORDCOUNT_SEEDS), "SEEDS ===\n")
for field in real_fields:
    field_name = field.replace("_clean", "")
    print(f"{field_name.upper()}")
    print(f"  Mean:   {rc_wordcount_df[f'{field}_mean'].mean():.2f} ± {rc_wordcount_df[f'{field}_mean'].std():.2f}")
    print(f"  Median: {rc_wordcount_df[f'{field}_median'].mean():.2f} ± {rc_wordcount_df[f'{field}_median'].std():.2f}")
    print(f"  Std:    {rc_wordcount_df[f'{field}_std'].mean():.2f} ± {rc_wordcount_df[f'{field}_std'].std():.2f}")
    print()

In [ ]:
# PART 7: JACCARD SIMILARITY ROBUSTNESS (10 seeds)
# Averages vocabulary overlap (top-50 terms) between real and synthetic across repeated samples

JACCARD_SEEDS = list(range(1, 11))
syn_fields = ["summary_clean_syn", "pros_clean_syn", "cons_clean_syn"]

rc_jaccard_runs = []

for seed in JACCARD_SEEDS:
    df_run = sample_matched(df_sample, group_cols, n=500, random_state=seed)

    row = {"seed": seed}
    for real_f, syn_f in zip(real_fields, syn_fields):
        result = compute_jaccard(df_run, synthetic_df, real_f, syn_f)
        field_name = real_f.replace("_clean", "")
        row[f"{field_name}_jaccard"] = result["jaccard_similarity"]
    rc_jaccard_runs.append(row)

rc_jaccard_df = pd.DataFrame(rc_jaccard_runs)

print("=== JACCARD SIMILARITY - AVERAGED ACROSS", len(JACCARD_SEEDS), "SEEDS ===\n")
for real_f in real_fields:
    field_name = real_f.replace("_clean", "")
    col = f"{field_name}_jaccard"
    print(f"{field_name.upper()}: {rc_jaccard_df[col].mean():.4f} ± {rc_jaccard_df[col].std():.4f} "
          f"(range: {rc_jaccard_df[col].min():.4f} - {rc_jaccard_df[col].max():.4f})")

In [ ]:
# Outlier (-1) rate check per combination - explains why synthetic topic counts may be inflated
# Confirms whether synthetic reviews cluster more easily than real reviews (fewer unassigned documents)
print("=== REAL: outlier (-1) counts and proportions per combination ===\n")
for combo, df in summary_topics_by_combo.items():
    total = df["Count"].sum()
    outliers = df[df["Topic"] == -1]["Count"].sum() if -1 in df["Topic"].values else 0
    print(f"{combo}: {outliers}/{total} outliers ({outliers/total*100:.1f}%)")

print("\n=== SYNTHETIC: outlier (-1) counts and proportions per combination ===\n")
for combo, df in summary_topics_by_combo_syn.items():
    total = df["Count"].sum()
    outliers = df[df["Topic"] == -1]["Count"].sum() if -1 in df["Topic"].values else 0
    print(f"{combo}: {outliers}/{total} outliers ({outliers/total*100:.1f}%)")